# 无参考图像质量评价系统 (NR-IQA)
## 基于语义保真度、技术质量、结构完整性的综合评估

---

## 项目概述

本 Notebook 实现了一个完整的**无参考图像质量评价(No-Reference Image Quality Assessment, NR-IQA)系统**。

### 核心特性

1. **文本语义解析** - 结构化提示词为四类语义要素
2. **语义保真度指标** - CLIP 相似度 + 关键词匹配率
3. **技术质量指标** - 清晰度、噪声、伪影
4. **结构完整性指标** - 边缘连续性、形状规则性
5. **加权综合质量指数** - 多维度综合评分

### 数学模型

```
Q_final = w₁ * S_fidelity + w₂ * Q_technical + w₃ * I_structural

其中:
- Q_final ∈ [0, 1]: 综合质量指数
- w₁ = 0.35: 语义保真度权重
- w₂ = 0.40: 技术质量权重
- w₃ = 0.25: 结构完整性权重
```

---

## 1. 环境设置和依赖导入

In [ ]:
# 标准库
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib import rcParams
import json
import os
from pathlib import Path

# 图像处理
from PIL import Image
import cv2

# 设置中文字体支持
rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
rcParams['axes.unicode_minus'] = False

# 导入本地模块
import sys
sys.path.insert(0, './src')

from quality_index_model import NRIQAModel
from text_semantic_parser import TextSemanticParser
from semantic_fidelity import SemanticFidelityAnalyzer
from technical_quality import TechnicalQualityAnalyzer
from structural_integrity import StructuralIntegrityAnalyzer

print("✓ 所有依赖导入成功")
print(f"NumPy 版本: {np.__version__}")

## 2. 初始化 NR-IQA 模型

In [ ]:
# 创建 NR-IQA 模型实例
# 权重配置说明:
# - weight_semantic=0.35: 语义一致性权重（文本-图像匹配度）
# - weight_technical=0.40: 技术质量权重（清晰度、噪声）
# - weight_structural=0.25: 结构完整性权重（边界、形状）

model = NRIQAModel(
    weight_semantic=0.35,
    weight_technical=0.40,
    weight_structural=0.25,
    use_clip=False  # 设置为 True 需要额外的 CLIP 模型
)

print("✓ NR-IQA 模型初始化完成")
print(f"\n模型权重配置:")
print(f"  - 语义保真度权重 (w₁): {model.weight_semantic}")
print(f"  - 技术质量权重 (w₂): {model.weight_technical}")
print(f"  - 结构完整性权重 (w₃): {model.weight_structural}")
print(f"  - 总计: {model.weight_semantic + model.weight_technical + model.weight_structural}")

## 3. 文本语义解析演示

In [ ]:
# 测试文本提示词
test_prompts = [
    "A beautiful sunset over the ocean with golden light",
    "A cat sitting on a table in a cozy room, watercolor style",
    "Mountain landscape with clear blue sky, cinematic lighting"
]

# 文本语义解析器
text_parser = TextSemanticParser()

print("=" * 70)
print("文本提示词语义解析演示")
print("=" * 70)

for idx, prompt in enumerate(test_prompts, 1):
    print(f"\n【示例 {idx}】")
    print(f"原始提示词: {prompt}")
    print()
    
    # 解析语义
    semantic_result = text_parser.parse(prompt)
    
    # 显示解析结果
    for category_name, element in semantic_result.items():
        print(f"  {category_name.upper():15s}: {element.tokens}")
        print(f"  {'置信度':15s}: {element.confidence:.2f}")
        print()

## 4. 生成测试图像并进行质量评估

In [ ]:
# 创建不同质量的测试图像
def create_test_image(quality_type='medium'):
    """
    创建不同质量等级的测试图像
    
    Args:
        quality_type: 'high', 'medium', 'low'
    """
    height, width = 512, 512
    
    if quality_type == 'high':
        # 高质量：平滑的渐变，低噪声
        image = np.zeros((height, width, 3), dtype=np.uint8)
        x = np.linspace(0, 255, width)
        y = np.linspace(0, 255, height)
        xx, yy = np.meshgrid(x, y)
        
        image[:, :, 0] = xx.astype(np.uint8)  # 红色渐变
        image[:, :, 1] = yy.astype(np.uint8)  # 绿色渐变
        image[:, :, 2] = ((xx + yy) / 2).astype(np.uint8)  # 蓝色混合
        
        # 添加低水平噪声
        noise = np.random.normal(0, 5, image.shape)
        
    elif quality_type == 'medium':
        # 中等质量：带有一些噪声的随机图像
        image = np.random.randint(50, 200, (height, width, 3), dtype=np.uint8)
        
        # 添加一些结构（圆形）
        cv2.circle(image, (256, 256), 100, (255, 0, 0), -1)
        
        # 中等水平噪声
        noise = np.random.normal(0, 15, image.shape)
        
    else:  # low
        # 低质量：高噪声，伪影
        image = np.random.randint(0, 256, (height, width, 3), dtype=np.uint8)
        
        # 添加块状伪影
        block_size = 32
        for i in range(0, height, block_size*2):
            for j in range(0, width, block_size*2):
                image[i:i+block_size, j:j+block_size] = 100
        
        # 高水平噪声
        noise = np.random.normal(0, 30, image.shape)
    
    # 添加噪声
    image = np.clip(image.astype(np.float32) + noise, 0, 255).astype(np.uint8)
    
    return image

# 创建测试图像
test_images = {
    'High Quality': create_test_image('high'),
    'Medium Quality': create_test_image('medium'),
    'Low Quality': create_test_image('low')
}

# 可视化
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Test Images with Different Quality Levels', fontsize=14, fontweight='bold')

for idx, (title, image) in enumerate(test_images.items()):
    axes[idx].imshow(image)
    axes[idx].set_title(title, fontsize=12)
    axes[idx].axis('off')

plt.tight_layout()
plt.show()

print("✓ 测试图像创建完成")

## 5. 对不同质量图像进行评估

In [ ]:
# 评估每个测试图像
test_prompt = "A beautiful sunset over the ocean with golden light"

results = {}

print("=" * 70)
print("开始图像质量评估")
print("=" * 70)
print(f"\n测试提示词: {test_prompt}\n")

for quality_level, image in test_images.items():
    print(f"正在评估: {quality_level}...")
    result = model.evaluate(image, test_prompt)
    results[quality_level] = result
    print(f"  ✓ 完成")

print("\n✓ 所有图像评估完成")

## 6. 结果可视化和比较分析

In [ ]:
# 提取综合质量分数
quality_levels = list(results.keys())
overall_scores = [results[level]['overall_quality'] for level in quality_levels]
semantic_scores = [results[level]['semantic_fidelity']['semantic_fidelity'] for level in quality_levels]
technical_scores = [results[level]['technical_quality']['technical_quality'] for level in quality_levels]
structural_scores = [results[level]['structural_integrity']['structural_integrity'] for level in quality_levels]

# 绘制综合对比图
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('NR-IQA 综合评估结果对比', fontsize=16, fontweight='bold')

# 1. 综合质量指数
ax1 = axes[0, 0]
colors = ['green' if s > 0.7 else 'orange' if s > 0.4 else 'red' for s in overall_scores]
ax1.bar(quality_levels, overall_scores, color=colors, alpha=0.7, edgecolor='black')
ax1.set_ylabel('Score', fontsize=11)
ax1.set_title('综合质量指数 (Overall Quality)', fontsize=12, fontweight='bold')
ax1.set_ylim([0, 1])
ax1.axhline(y=0.8, color='g', linestyle='--', alpha=0.5, label='优秀 (Excellent)')
ax1.axhline(y=0.6, color='orange', linestyle='--', alpha=0.5, label='良好 (Good)')
ax1.axhline(y=0.4, color='r', linestyle='--', alpha=0.5, label='中等 (Fair)')
ax1.legend()
for i, v in enumerate(overall_scores):
    ax1.text(i, v + 0.03, f'{v:.3f}', ha='center', fontweight='bold')

# 2. 三维度分数对比
ax2 = axes[0, 1]
x = np.arange(len(quality_levels))
width = 0.25
ax2.bar(x - width, semantic_scores, width, label='语义保真度', alpha=0.8)
ax2.bar(x, technical_scores, width, label='技术质量', alpha=0.8)
ax2.bar(x + width, structural_scores, width, label='结构完整性', alpha=0.8)
ax2.set_ylabel('Score', fontsize=11)
ax2.set_title('三维度指标对比', fontsize=12, fontweight='bold')
ax2.set_xticks(x)
ax2.set_xticklabels(quality_levels)
ax2.legend()
ax2.set_ylim([0, 1])

# 3. 技术质量细节
ax3 = axes[1, 0]
sharpness = [results[level]['technical_quality']['sharpness'] for level in quality_levels]
noise = [results[level]['technical_quality']['noise'] for level in quality_levels]
artifacts = [results[level]['technical_quality']['artifacts'] for level in quality_levels]

ax3.plot(quality_levels, sharpness, marker='o', label='清晰度 (Sharpness)', linewidth=2)
ax3.plot(quality_levels, noise, marker='s', label='噪声 (Noise)', linewidth=2)
ax3.plot(quality_levels, artifacts, marker='^', label='伪影 (Artifacts)', linewidth=2)
ax3.set_ylabel('Score', fontsize=11)
ax3.set_title('技术质量细节指标', fontsize=12, fontweight='bold')
ax3.legend()
ax3.set_ylim([0, 1])
ax3.grid(alpha=0.3)

# 4. 结构完整性细节
ax4 = axes[1, 1]
edge_continuity = [results[level]['structural_integrity']['edge_continuity'] for level in quality_levels]
shape_regularity = [results[level]['structural_integrity']['shape_regularity'] for level in quality_levels]
object_completeness = [results[level]['structural_integrity']['object_completeness'] for level in quality_levels]

ax4.plot(quality_levels, edge_continuity, marker='o', label='边缘连续性', linewidth=2)
ax4.plot(quality_levels, shape_regularity, marker='s', label='形状规则性', linewidth=2)
ax4.plot(quality_levels, object_completeness, marker='^', label='对象完整性', linewidth=2)
ax4.set_ylabel('Score', fontsize=11)
ax4.set_title('结构完整性细节指标', fontsize=12, fontweight='bold')
ax4.legend()
ax4.set_ylim([0, 1])
ax4.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("✓ 结果可视化完成")

## 7. 生成详细评估报告

In [ ]:
# 为每个图像生成详细报告
print("\n" + "=" * 70)
print("详细评估报告")
print("=" * 70)

for quality_level in quality_levels:
    result = results[quality_level]
    report = model.generate_report(result)
    print(report)

## 8. 数学模型详细解释

In [ ]:
print("=" * 70)
print("NR-IQA 数学模型详细解释")
print("=" * 70)

print("""
【1. 综合质量指数 (Overall Quality Index)】

数学定义:
    Q_final = w₁ * S_fidelity + w₂ * Q_technical + w₃ * I_structural

参数说明:
    - Q_final ∈ [0, 1]:        综合质量指数
    - w₁ = 0.35:               语义保真度权重
    - w₂ = 0.40:               技术质量权重  
    - w₃ = 0.25:               结构完整性权重
    - S_fidelity ∈ [0, 1]:     语义保真度分数
    - Q_technical ∈ [0, 1]:    技术质量分数
    - I_structural ∈ [0, 1]:   结构完整性分数

物理意义:
    高质量(>0.8):  可用于商业/专业应用
    良好(0.6-0.8): 适合网络/社交媒体发布
    中等(0.4-0.6): 需要优化改进
    较差(<0.4):    不建议使用

---

【2. 语义保真度 (Semantic Fidelity)】

数学定义:
    S_fidelity = α * sim_CLIP + β * KMR
    其中: α = 0.7, β = 0.3

子指标:
    - sim_CLIP ∈ [0, 1]:  CLIP 图像-文本语义相似度
        计算方法: 余弦相似度 = (I · T) / (||I|| * ||T||)
        范围标准化: (cosine_sim + 1) / 2 → [0, 1]
    
    - KMR ∈ [0, 1]:       关键词匹配率
        计算方法: 提示词中的语义关键词在图像中的匹配比例
        权重分配: subject(0.4) > attributes(0.3) > scene(0.2) > style(0.1)

物理意义:
    - 高值: 图像忠实地实现了文本提示的语义意图
    - 低值: 图像与文本提示的语义偏离较大

---

【3. 技术质量 (Technical Quality)】

数学定义:
    Q_technical = α₁*Sharpness + α₂*(1-Noise) + α₃*(1-Artifacts)
    其中: α₁=0.4, α₂=0.3, α₃=0.3

子指标:
    
    a) 清晰度 (Sharpness) ∈ [0, 1]
       定义: Sharpness = sigmoid(var(∇²I) / 1000)
       其中 ∇² 是拉普拉斯算子
       物理意义: 图像梯度的方差，反映边界清晰程度
    
    b) 噪声 (Noise) ∈ [0, 1]
       定义: Noise = E_high_freq / E_total
       计算: 高频分量占总能量的比例（FFT分析）
       物理意义: 高频能量越多，噪声越严重
    
    c) 伪影 (Artifacts) ∈ [0, 1]
       定义: Artifacts = (边缘断裂 + 频域异常) / 2
       物理意义: 不自然的块状伪影和边界不连贯

---

【4. 结构完整性 (Structural Integrity)】

数学定义:
    I_structural = β₁*EdgeContinuity + β₂*ShapeRegularity + 
                   β₃*Completeness + β₄*Smoothness
    其中: β₁=0.3, β₂=0.25, β₃=0.25, β₄=0.2

子指标:
    
    a) 边缘连续性 (Edge Continuity) ∈ [0, 1]
       定义: 最大连通边缘长度 / 总边缘长度
       物理意义: 对象边界的连贯程度，高值表示边界清晰无断裂
    
    b) 形状规则性 (Shape Regularity) ∈ [0, 1]
       定义: Polsby-Popper 圆形度指标
             Regularity = 1 / (1 + (4πA/P²))
       其中: A = 面积, P = 周长
       物理意义: 完全圆形为1，不规则形状为<0.5
    
    c) 对象完整性 (Object Completeness) ∈ [0, 1]
       定义: 无孔洞对象数 / 总对象数
       物理意义: 对象内部孔洞面积 < 对象面积的10%时认为完整
    
    d) 边界平滑度 (Boundary Smoothness) ∈ [0, 1]
       定义: 1 - (高曲率边界比例)
       物理意义: 平滑曲线为高值，锯齿状边界为低值

---

【权重配置指南】

不同应用场景的权重建议:

1. 文本驱动生成 (Text-to-Image):
   w₁=0.5, w₂=0.3, w₃=0.2
   -> 强调语义一致性

2. 通用图像评估:
   w₁=0.35, w₂=0.40, w₃=0.25 (默认)
   -> 平衡三个维度

3. 物体检测任务:
   w₁=0.2, w₂=0.3, w₃=0.5
   -> 强调边界和结构

4. 艺术创意生成:
   w₁=0.3, w₂=0.35, w₃=0.35
   -> 均衡创意和质量
""")

print("=" * 70)

## 9. 批量处理和导出结果

In [ ]:
# 示例：如何批量处理多张图像
print("=" * 70)
print("批量图像处理演示")
print("=" * 70)

# 创建多个测试图像
test_batch = [
    create_test_image('high'),
    create_test_image('medium'),
    create_test_image('low')
]

test_prompts_batch = [
    "A beautiful sunset",
    "A cozy room",
    "Mountain landscape"
]

# 批量评估
print(f"\n正在处理 {len(test_batch)} 张图像...")
batch_results = model.batch_evaluate(test_batch, test_prompts_batch)
print("✓ 批量处理完成\n")

# 显示批量结果摘要
print("批量处理结果摘要:")
print("-" * 70)
for idx, result in enumerate(batch_results, 1):
    print(f"\n图像 {idx}:")
    print(f"  提示词: {test_prompts_batch[idx-1]}")
    print(f"  综合质量: {result['overall_quality']:.4f}")
    print(f"  等级: {result['quality_interpretation']['level']}")
    print(f"  建议: {result['quality_interpretation']['recommendation']}")

## 10. 自定义权重配置实验

In [ ]:
# 对比不同权重配置的影响
print("=" * 70)
print("权重配置实验")
print("=" * 70)

weight_configs = {
    '文本驱动生成\n(w₁=0.5, w₂=0.3, w₃=0.2)': (0.5, 0.3, 0.2),
    '通用配置\n(w₁=0.35, w₂=0.4, w₃=0.25)': (0.35, 0.4, 0.25),
    '物体检测\n(w₁=0.2, w₂=0.3, w₃=0.5)': (0.2, 0.3, 0.5),
    '艺术创意\n(w₁=0.3, w₂=0.35, w₃=0.35)': (0.3, 0.35, 0.35)
}

test_image = create_test_image('medium')

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('不同权重配置对评分的影响', fontsize=16, fontweight='bold')

for idx, (config_name, (w1, w2, w3)) in enumerate(weight_configs.items()):
    ax = axes[idx // 2, idx % 2]
    
    # 创建临时模型
    temp_model = NRIQAModel(weight_semantic=w1, weight_technical=w2, weight_structural=w3)
    result = temp_model.evaluate(test_image, test_prompts[0])
    
    # 提取分数
    scores = {
        '语义保真度': result['semantic_fidelity']['semantic_fidelity'],
        '技术质量': result['technical_quality']['technical_quality'],
        '结构完整性': result['structural_integrity']['structural_integrity'],
        '综合质量': result['overall_quality']
    }
    
    # 绘制
    colors_map = {'语义保真度': '#FF6B6B', '技术质量': '#4ECDC4', '结构完整性': '#45B7D1', '综合质量': '#FFA07A'}
    colors = [colors_map[k] for k in scores.keys()]
    
    bars = ax.bar(scores.keys(), scores.values(), color=colors, alpha=0.7, edgecolor='black')
    ax.set_title(config_name, fontsize=11, fontweight='bold')
    ax.set_ylim([0, 1])
    ax.set_ylabel('Score')
    
    # 在柱子上显示数值
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
               f'{height:.3f}', ha='center', va='bottom', fontweight='bold')
    
    # 权重信息
    info_text = f'w₁={w1}\nw₂={w2}\nw₃={w3}'
    ax.text(0.95, 0.05, info_text, transform=ax.transAxes,
           fontsize=9, verticalalignment='bottom', horizontalalignment='right',
           bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')

plt.tight_layout()
plt.show()

print("\n✓ 权重配置实验完成")

## 11. 总结和建议

In [ ]:
print("=" * 70)
print("NR-IQA 系统总结")
print("=" * 70)

print("""
【系统特点】

1. 多维度评估
   - 语义保真度: 评估图像与文本提示的一致性
   - 技术质量: 评估清晰度、噪声、伪影
   - 结构完整性: 评估对象边界和形状质量

2. 可解释性强
   - 每个指标都有明确的数学定义
   - 支持详细的分项报告
   - 权重配置灵活，可针对不同应用场景调整

3. 无参考设计
   - 不依赖参考图像
   - 适合评估生成式模型的输出
   - 支持实时评估

【使用建议】

1. 权重配置选择
   - 根据应用场景选择合适的权重配置
   - 文本驱动生成: 提高语义权重
   - 物体检测: 提高结构权重
   - 通用评估: 使用默认配置

2. 阈值设置
   高质量 (>0.8):   用于商业/专业应用
   良好 (0.6-0.8):  用于网络/社交媒体
   中等 (0.4-0.6):  需要优化改进
   较差 (<0.4):     不建议使用

3. 批量处理
   - 使用 batch_evaluate() 处理多张图像
   - 使用 generate_report() 生成详细报告
   - 使用 export_results() 导出结果

【改进方向】

1. 可以整合实时的 CLIP 模型，提高语义保真度的准确性
2. 可以添加颜色一致性评估（色差ΔE等）
3. 可以针对特定领域（医学影像、遥感等）进行微调
4. 可以添加感知质量指标（SSIM、LPIPS等）

【文献参考】

[1] Radford et al., "Learning Transferable Models for Computer Vision Tasks"
    (CLIP, 2021)
[2] Wang et al., "Image Quality Assessment: From Error Visibility to Structural
    Similarity" (SSIM, 2004)
[3] Sheikh & Bovik, "Image Information and Visual Quality" (VIF, 2006)
[4] Gu et al., "Perceptual Quality Assessment of Images" (Survey)

---

✓ NR-IQA 系统演示完成！

更多信息请访问: https://github.com/FFbond-hao/NR-IQA-Semantic-Quality
""")

print("=" * 70)